# Qwen3-TTS single-speaker **LoRA** fine-tune

Same data and pipeline as `qwen3tts_finetune_colab.ipynb`; only the training step
differs. Upstream ships no PEFT path, so step 3 injects one.

**Read `../FINETUNING.md` before changing anything here.** It documents every
patch, the upstream facts they rest on (with line numbers pinned to clone commit
`022e286`), and the mistakes that produced them.

## Run in order — one pass, top to bottom

| | cell | notes |
|---|---|---|
| 1 | **1** GPU check | **set `DATASET_REPO` and `SPEAKER`** — cell asserts you did |
| 2 | **1b** resource probe | defines `res()`, used throughout |
| 3 | **2** install + clone | ends `GATE 2 OK` |
| 4 | **3** patch | ends `GATE 3 OK`; **re-runnable, patches from pristine** |
| 5 | **3b** base download | ends `GATE 3b OK -- base is 2.3xGiB` |
| 6 | **4** dataset pull | asks for a **read** token; ends `GATE 4 OK -- 167 clips, 20.0 min, all 24kHz` |
| 7 | **5** tokenise | then **5b** → `167 rows ... code len ~93` |
| 8 | **6** train | ~25 min. Watch `bad_grads: 0` and a **flat** `peak host RSS` |
| 9 | **7** write `gen.py` | no output but the filename |
| 10 | **7a** generate | the 2×2; **`STOPPED on codec_eos` is the number that matters** |
| 11 | **7c** listen | before scoring anything |
| 12 | **7d** held-out + baseline | ICL (55.1% bar) and x-vector (39.1%) |
| 13 | **7e-pre** → **7e** | env gate, then WER |
| 14 | **7f** curve, **8** push | |

**Skip 7b** unless 7a's *fp32* runs also fail. Its own header explains why the
hypothesis it tests is dead.

## What this run is testing

The previous three runs trained a **shifted objective** — see §3(i) of
`FINETUNING.md`. `ForCausalLMLoss` shifts labels internally and upstream pre-shifts
them too, so every frame was predicted two positions ahead and `codec_eos` was
supervised on the wrong frame. The loss fell convincingly (12.30 → 2.32) and the
audio was unusable. Patch (i) fixes both call sites. **This is the first run on the
correct objective.**

## Two things to know about the numbers

**LoRA (here):** base frozen, 23.8M trainable adapter params (2.5592%). Measured
peak **9.0GB** GPU of 15.0 on a T4 — fp32 throughout, because fp16 breaks
generation (`multinomial` asserts on non-finite probs) and buys ~5 min.

**The speaker embedding is not learned by gradient descent** in either version. It
comes from `speaker_encoder(ref_mels)` and is written into
`codec_embedding.weight[3000]` at save time — so LoRA changes how the voice is
*modelled*, not how it is *captured*. Its norm is ~9.9 against ~0.49 for real codec
rows; that is correct, not corruption.


In [ ]:
# 1. Check the GPU you actually got
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| bf16 supported:", torch.cuda.is_bf16_supported())

# --- SET THESE TO YOUR OWN ---------------------------------------------------
import os
DATASET_REPO = "your-username/your-voice-tts"   # private HF dataset repo, step 4
SPEAKER      = "myvoice"                        # name the voice registers under
os.environ["SPEAKER"] = SPEAKER                 # gen.py (step 7) reads it from env
# -----------------------------------------------------------------------------
# Left as a placeholder on purpose: this repo is public and the dataset repo id
# is not. Checked here rather than at step 4, which is a pip install, a clone
# and a patch later.
assert "your-username" not in DATASET_REPO, "set DATASET_REPO to your own repo"


In [ ]:
# 1b. Resource probe. Pure measurement -- it reads counters and changes nothing
#     about training, so numbers from a run with logging are directly comparable
#     to one without.
import subprocess, shutil, os
def res(tag=""):
    try:
        u, t = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True).strip().split("\n")[0].split(",")
        gpu = f"GPU {int(u)/1024:.1f}/{int(t)/1024:.1f}GB"
    except Exception:
        gpu = "GPU n/a"
    try:
        import psutil
        v = psutil.virtual_memory()
        ram = f"RAM {(v.total-v.available)/2**30:.1f}/{v.total/2**30:.1f}GB"
    except Exception:
        ram = "RAM n/a"
    d = shutil.disk_usage("/content")
    print(f"[res] {tag:24s} {gpu} | {ram} | DISK {d.used/2**30:.1f}/{d.total/2**30:.1f}GB")
res("baseline")


In [ ]:
# 2. Install + clone. Takes a few minutes.
# peft belongs here, not in cell 4: GATE 2 below imports it. qwen-tts does not
# depend on it (pyproject lists transformers/accelerate/gradio/librosa/torchaudio/
# soundfile/sox/onnxruntime/einops) and nothing under qwen_tts/ imports it, so on
# a fresh Colab it is simply absent.
!pip -q install -U qwen-tts accelerate peft
!git clone -q https://github.com/QwenLM/Qwen3-TTS.git
%cd /content/Qwen3-TTS/finetuning
# Colab preinstalls torchao 0.10.0. peft walks every dispatcher in
# lora/model.py:_create_new_module, and dispatch_torchao calls
# is_torchao_available(), which RAISES on a version below 0.16.0 instead of
# returning False (import_utils.py:147). So a library this pipeline never uses
# kills get_peft_model. Uninstalling is the fix, not upgrading: the function
# short-circuits to False when find_spec('torchao') is None, and pulling
# torchao >0.16 would drag a torch upgrade onto a working CUDA stack.
# No kernel restart needed -- sft_12hz.py runs as a subprocess and imports fresh.
# Done HERE, before GATE 2 imports peft: the failure we hit was at
# get_peft_model rather than at import, but that is not worth betting a cell on
# when the uninstall is free and cell 3 always precedes cell 4 anyway (a fresh
# runtime has no clone for cell 4 to patch).
!pip -q uninstall -y torchao


# GATE 2. Everything downstream assumes this cwd and this clone. A failed pip or
# a rate-limited clone otherwise surfaces as a confusing error two cells later.
import importlib, pathlib
assert pathlib.Path.cwd() == pathlib.Path("/content/Qwen3-TTS/finetuning"), pathlib.Path.cwd()
for f in ["sft_12hz.py", "prepare_data.py", "dataset.py"]:
    assert pathlib.Path(f).exists(), f"clone incomplete: {f} missing"
import qwen_tts, accelerate, peft   # noqa: F401  -- import, not pip show: a broken
                                    # install can leave the dist-info behind
print("qwen_tts", qwen_tts.__version__ if hasattr(qwen_tts, "__version__") else "ok",
      "| accelerate", accelerate.__version__, "| peft", peft.__version__)
print("GATE 2 OK")


In [ ]:
# 3. PATCH sft_12hz.py. Five things upstream assumes that are not true here.
# Reset first: an earlier run of this cell already edited the file, so the
# replaces below would silently miss their targets. Always patch from pristine.
!git -C /content/Qwen3-TTS checkout -- finetuning/sft_12hz.py
!pip -q install -U bitsandbytes

import torch, pathlib
p = pathlib.Path("sft_12hz.py"); s = p.read_text()

# a) flash_attention_2 needs Ampere+ (sm_80). sdpa ships with torch and works anywhere.
s = s.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# b) log_with="tensorboard" needs a logging_dir. init_trackers is never called and
#    nothing ever .log()s, so this is dead config -> drop it.
s = s.replace(', log_with="tensorboard"', '')

# c) T4 is Turing: no native bf16 (is_bf16_supported() says True only via emulation),
#    so upstream's bf16 has to go. PURE FP32, not fp16 autocast.
#
#    fp16 was the obvious substitute and is the wrong one. Weights must be fp32
#    regardless -- accelerator.prepare wraps the optimizer in a GradScaler, and
#    fp16 weights + a scaler raises "Attempting to unscale FP16 gradients", AMP
#    needs fp32 masters. So fp16 buys only autocast matmuls on tensor cores:
#    ~5-7 min off a 13-min run. What it costs is the whole GradScaler failure
#    mode -- on an fp16 overflow accelerate SKIPS optimizer.step() and halves the
#    scale, silently, while the step counter keeps counting -- plus the
#    instrumentation to prove that is not happening. Delete the failure class
#    instead of measuring it.
#
#    It also matches inference, which gen.py now forces to fp32 on pre-Ampere,
#    removing one axis of difference between training and generation.
#
#    MEMORY: fp16 autocast peaked at 8.8GB of 14.56GB usable. Static is ~3.9GB
#    (fp32 weights 3.7 + LoRA grads + 8-bit states), so ~4.9GB of that was
#    activations and workspace, and only that share grows here -> expect
#    ~11-13GB. Tight. If it OOMs, drop --batch_size to 1 and pass
#    --gradient_accumulation_steps 8 for the same effective batch (upstream
#    hardcodes 4 at line 44, so that needs a patch here too).
if torch.cuda.get_device_capability()[0] < 8:
    s = s.replace('mixed_precision="bf16"', 'mixed_precision="no"')
    s = s.replace("torch_dtype=torch.bfloat16", "torch_dtype=torch.float32")

# d) UPSTREAM BUG. text_embedding is nn.Embedding(vocab, text_hidden_size=2048) in
#    both sizes, but talker hidden_size is 2048 on the 1.7B and 1024 on the 0.6B.
#    The model has a text_projection MLP (modeling_qwen3_tts.py:1575) that resizes
#    2048 -> hidden_size, and every inference path uses it; sft_12hz.py:89 does not.
#    On the 1.7B that is invisible (2048->2048); on the 0.6B it is a shape error.
#    Project first, then mask -- the MLP has bias=True, so masking last is required
#    to keep padded positions at zero.
old = "input_text_embedding = model.talker.model.text_embedding(input_text_ids) * text_embedding_mask"
new = ("input_text_embedding = model.talker.text_projection(\n"
       "                    model.talker.model.text_embedding(input_text_ids)) * text_embedding_mask")
assert old in s, "line 89 not found - upstream may have fixed this"
s = s.replace(old, new)

# e) OOM. This checkpoint is 914.6M params, not 0.6B (talker.model 754.8M +
#    code_predictor 141.6M + the rest). Full fp32 AdamW = weights 3.7 + grads 3.7
#    + states 7.3 = 14.6GB, against 14.56GB usable on a T4. Batch size is
#    irrelevant -- optimizer state does not depend on it. 8-bit Adam keeps the
#    full fine-tune and drops states 7.3GB -> 1.8GB.
if torch.cuda.get_device_properties(0).total_memory < 24e9:
    s = s.replace("from torch.optim import AdamW",
                  "from bitsandbytes.optim import AdamW8bit as AdamW")

# f) LoRA. Wrap the talker (the only module that gets gradients) and train ~2% of
#    the weights. peft forwards ONE-HOP attribute access to the base module, so
#    model.talker.text_projection / .code_predictor still resolve; only
#    state_dict() key names change, handled in (g). A `.model` hop does not
#    survive the wrapper -- see (f2).
s = s.replace(
    "    config = AutoConfig.from_pretrained(MODEL_PATH)",
    "    from peft import LoraConfig, get_peft_model\n"
    "    lora = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',\n"
    "                      target_modules=['q_proj','k_proj','v_proj','o_proj',\n"
    "                                      'gate_proj','up_proj','down_proj'])\n"
    "    qwen3tts.model.talker = get_peft_model(qwen3tts.model.talker, lora)\n"
    "    qwen3tts.model.talker.print_trainable_parameters()\n"
    "    config = AutoConfig.from_pretrained(MODEL_PATH)")

#    Only feed the optimizer parameters that actually require grad. Plain AdamW
#    would skip the frozen ones anyway (grad is None), but bnb's 8-bit AdamW
#    allocates per registered param, which would waste the saving.
s = s.replace(
    "optimizer = AdamW(qwen3tts.model.parameters(), lr=args.lr, weight_decay=0.01)",
    "optimizer = AdamW([p for p in qwen3tts.model.parameters() if p.requires_grad],\n"
    "                      lr=args.lr, weight_decay=0.01)")

# g) Save a MERGED full checkpoint, so inference needs no peft and the existing
#    epoch-save logic (copytree + full safetensors) works untouched.
#    merge_adapter/unmerge_adapter fold in place -- no deepcopy of 3.6GB.
#    TWO renames are needed, not one. peft registers the wrapped module as a
#    submodule (lora/layer.py:129 `self.base_layer = base_layer`), so every
#    targeted projection saves as q_proj.base_layer.weight. Stripping only the
#    talker.base_model.model. prefix leaves that infix, the base model never
#    finds q_proj.weight, from_pretrained leaves those layers at init and gen.py
#    emits noise -- after a clean run that raised nothing. The merged weights ARE
#    in base_layer.weight; only the key name is wrong.
#
#    .clone() IS LOAD-BEARING. Tensor.to('cpu') returns self when the tensor is
#    already on cpu, so without it this dict holds REFERENCES to the merged
#    weights and the unmerge_adapter() below rewinds the very values being
#    saved -- a checkpoint numerically identical to the base model, exactly the
#    failure the key-name fix above prevents, reached from the other side. On
#    cuda .to('cpu') already copies, so this is latent on Colab and fires the
#    moment training runs cpu-only, which this project has fallen back to before
#    (ft_2_cpu.wav was the only usable output of the 9.5% run). The asserts
#    below cannot catch it: the key names are right, the numbers are not.
s = s.replace(
    '            state_dict = {k: v.detach().to("cpu") for k, v in unwrapped_model.state_dict().items()}',
    "            unwrapped_model.talker.merge_adapter()\n"
    "            state_dict = {k: v.detach().to('cpu').clone()\n"
    "                          for k, v in unwrapped_model.state_dict().items()\n"
    "                          if 'lora_' not in k}\n"
    "            state_dict = {k.replace('talker.base_model.model.', 'talker.')\n"
    "                           .replace('.base_layer.', '.'): v\n"
    "                          for k, v in state_dict.items()}\n"
    "            assert not [k for k in state_dict if 'base_layer' in k or 'lora' in k], \\\n"
    "                'peft key names survived into the checkpoint'\n"
    "            assert 'talker.model.codec_embedding.weight' in state_dict, \\\n"
    "                'line 155 will KeyError'\n"
    "            unwrapped_model.talker.unmerge_adapter()")
assert "merge_adapter" in s and "get_peft_model" in s, "LoRA patches did not apply"

# g2) ATOMIC CHECKPOINT. The save is copytree(MODEL_PATH) -> rewrite config.json
#     -> save_file(model.safetensors), ~30s. Die anywhere in that window and the
#     directory is FULL SIZE, has a valid config.json, reports the speaker from
#     get_supported_speakers(), loads clean, generates clean -- and holds the
#     BASE weights, because copytree brought the base model.safetensors along and
#     save_file had not overwritten it yet. Nothing is truncated, so a size
#     heuristic cannot see it. That is a third route to a base-identical
#     checkpoint, after the .base_layer key names and the .to('cpu') aliasing,
#     and the quietest of the three.
#
#     Build in <name>.tmp and rename at the end. rename(2) within one filesystem
#     is atomic, so checkpoint-epoch-N either does not exist or is complete --
#     which also retires the >=0.9*max size filters in 7a/7b.
#
#     Ceiling: the rmtree of an existing final_dir is a ~0.1s window in which an
#     interrupt leaves a half-deleted old checkpoint. Clearing output/ before the
#     run (step 6) means it never has one to delete.
#
# g2c) HOST RAM. THIS IS WHAT KILLED BOTH RUNS. `state_dict` is a plain local in
#      train() and nothing ever frees it, so it survives from one save to the
#      next -- ~3.62GB of fp32 CPU tensors (905.8M non-LoRA params x 4 bytes).
#      At the NEXT save, `state_dict = {...}` builds the replacement in full
#      BEFORE rebinding the name, so both copies exist at once: measured 9.02GB
#      after the first save, + 3.62GB = 12.64GB against a 12.7GB Colab cap.
#
#      The signature is exact. BOTH runs of this project died at their SECOND
#      save and never a first: run 1 wrote checkpoint-epoch-0 then died in
#      epoch-1's copytree; run 2 wrote checkpoint-epoch-3 then died in epoch-7's.
#      A first save has one state_dict live, a second has two.
#
#      This is a cost of patch (c) that I did not foresee: upstream's bf16 state
#      dict is 1.81GB, so even two of them fit. Choosing fp32 weights doubled it.
#      del + gc.collect() caps it at one copy and the peak drops to ~9GB.
s = s.replace(
    '            output_dir = os.path.join(args.output_model_path, f"checkpoint-epoch-{epoch}")',
    '            final_dir = os.path.join(args.output_model_path, f"checkpoint-epoch-{epoch}")\n'
    '            output_dir = final_dir + ".tmp"\n'
    '            shutil.rmtree(final_dir, ignore_errors=True)\n'
    '            shutil.rmtree(output_dir, ignore_errors=True)')
s = s.replace('            save_file(state_dict, save_path)',
              '            save_file(state_dict, save_path)\n'
              '            os.rename(output_dir, final_dir)\n'
              '            accelerator.print(f"saved {final_dir}", flush=True)\n'
              '            del state_dict, weight\n'
              '            gc.collect()\n'
              '            accelerator.print(f"  peak host RSS so far: "\n'
              '                f"{__import__(\'resource\').getrusage(0).ru_maxrss/2**20:.2f}GB",\n'
              '                flush=True)')
s = s.replace("import os", "import gc, os", 1)
assert "del state_dict, weight" in s and "import gc" in s, "(g2c) did not apply"
assert "os.rename(output_dir, final_dir)" in s, "(g2) did not apply"

# g2b) DO NOT COPY THE BASE WEIGHTS ONLY TO OVERWRITE THEM. copytree brings the
#      base model.safetensors (1.70GiB) and save_file replaces it seconds later,
#      so every save writes 1.70GiB for nothing: ~6.0GiB per checkpoint instead
#      of ~4.3GiB. Measured on this run, a save took 2m44s (last step of epoch 3
#      at 04:55:35, first of epoch 4 at 04:58:19) -- a silent window with no
#      output at all, which is where BOTH interrupts of this project have landed.
#      Cutting ~28% of the bytes cuts the window proportionally.
#
#      The callback must fire ONLY at the top level: speech_tokenizer has its own
#      model.safetensors (0.64GiB) which nothing else writes, and skipping it
#      would produce a checkpoint that cannot load.
s = s.replace(
    "            shutil.copytree(MODEL_PATH, output_dir, dirs_exist_ok=True)",
    "            _skip = lambda d, names: ([\"model.safetensors\"]\n"
    "                if os.path.abspath(d) == os.path.abspath(MODEL_PATH) else [])\n"
    "            shutil.copytree(MODEL_PATH, output_dir, dirs_exist_ok=True, ignore=_skip)")
assert "ignore=_skip" in s, "(g2b) did not apply"

# g3) SAVE EVERY 4th EPOCH, plus the last. Upstream saves unconditionally, which
#     was fine at 3 epochs and is not at 16: each checkpoint is ~4.4GB (3.7GB
#     safetensors plus the 682MB speech_tokenizer that copytree drags in), so 16
#     would want ~70GB against ~55GB free and the run would die on disk in the
#     back half. Every 4th gives checkpoints at epochs 3/7/11/15 -- ~17.6GB, and
#     four points to hear the voice move through rather than one endpoint.
#
#     Costs ~13 min of exposure: die before epoch 3 and there is nothing to load.
#     Acceptable against losing the run to a full disk at epoch 12.
s = s.replace(
    "        if accelerator.is_main_process:",
    "        if accelerator.is_main_process and (\n"
    "                (epoch + 1) % 4 == 0 or epoch == num_epochs - 1):")
assert "epoch == num_epochs - 1" in s, "(g3) did not apply"

# f2) PEFT SHADOWS `.model`. get_peft_model returns a PeftModel whose own .model
#     IS the LoraModel wrapper, so `model.talker.model.text_embedding` (lines 89
#     and 90) stops reaching the talker's inner model -- it lands on LoraModel and
#     raises AttributeError at the first training step, ~15 min into the run.
#     One-hop forwarding rescues .text_projection and .code_predictor (lines
#     96/111); it cannot rescue a `.model` hop. So resolve the unwrapped talker
#     once and use it for the two embedding lookups. Line 100's
#     `model.talker(...)` deliberately stays wrapped -- that is the call LoRA has
#     to intercept for anything to train at all.
anchor = "                input_codec_ids = input_ids[:, :, 1]"
assert anchor in s, "line 87 anchor not found - upstream may have moved"
s = s.replace(anchor, anchor + "\n"
              "                _tk = getattr(model.talker, 'get_base_model',\n"
              "                              lambda: model.talker)()")
s = s.replace("model.talker.text_projection(", "_tk.text_projection(")
s = s.replace("model.talker.model.text_embedding", "_tk.model.text_embedding")
s = s.replace("model.talker.model.codec_embedding", "_tk.model.codec_embedding")
assert "model.talker.model." not in s, "a .model hop survived; it will break under PEFT"
assert "outputs = model.talker(" in s, "the wrapped talker call must survive"

# h) LOGGING. Upstream prints only loss, every 10 steps, unbuffered nowhere. Two
#    things it already computes and discards:
#      - clip_grad_norm_ RETURNS the grad norm. Exploding or collapsing gradients
#        are invisible without it, and it is the honest health signal here.
#      - a non-finite loss shows up only as a weird number in a print.
#
#    NOT LOGGING GradScaler.get_scale(). Under (c) there is no scaler at all now,
#    but the number would have been misleading even under fp16: it starts at
#    65536, halves on overflow, and doubles only after growth_interval=2000
#    CONSECUTIVE good steps. This run is 4 x 84 = 336 steps, so the growth
#    interval is never reached and the scale can only fall or stay flat, by
#    construction. The early drops are the scaler calibrating -- designed
#    behaviour -- and look identical to the pathology. Alarming and correct at
#    the same time is the worst kind of metric.
#
#    `step` IS NOT A WEIGHT UPDATE. Line 71 wraps the body in
#    `with accelerator.accumulate(model)` and line 117 guards the clip with
#    `if accelerator.sync_gradients`, so with gradient_accumulation_steps=4
#    (line 44, hardcoded, no CLI arg) optimizer.step() is a no-op 3 times out of
#    4. 167 clips at batch 2 = 84 dataloader steps per epoch but only 21 real
#    updates. Upstream's counter is the dataloader one, so "Step 80" was 20
#    updates in, and reading the loss curve against it overstates the run by 4x.
#    _opt is the number that matters for whether the adapters have moved.
#
#    Both counters live inside the sync_gradients block -- that is the only place
#    a grad norm exists and the only place a step can be skipped, so they are 1:1
#    with real updates. Counting outside it would tally each norm 4 times and, on
#    non-sync steps, read a stale value from the previous sync.
s = s.replace("    model.train()",
              "    model.train()\n    _gnorm = None\n    _bad = 0\n    _opt = 0")

s = s.replace("                    accelerator.clip_grad_norm_(model.parameters(), 1.0)",
              "                    _gnorm = accelerator.clip_grad_norm_(model.parameters(), 1.0)\n"
              "                    _opt += 1\n"
              "                    if not torch.isfinite(_gnorm):\n"
              "                        _bad += 1")

_old_log = ('            if step % 10 == 0:\n'
            '                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")')
_new_log = (
    '            if not torch.isfinite(loss):\n'
    '                accelerator.print(f"!! NON-FINITE LOSS epoch {epoch} step {step}: {loss.item()}")\n'
    '            if step % 10 == 0:\n'
    '                import time as _t\n'
    '                _g = "n/a" if _gnorm is None else f"{float(_gnorm):.3f}"\n'
    '                accelerator.print(\n'
    '                    f"[{_t.strftime(\'%H:%M:%S\')}] Epoch {epoch} | Step {step} "\n'
    '                    f"| Loss: {loss.item():.4f} | grad_norm: {_g} "\n'
    '                    f"| updates: {_opt} | bad_grads: {_bad}",\n'
    '                    flush=True)')
assert _old_log in s, "log line not found - upstream may have changed it"
s = s.replace(_old_log, _new_log)
assert "_gnorm = accelerator.clip_grad_norm_" in s and "bad_grads" in s, "(h) did not apply"
assert "_opt += 1" in s, "(h) optimizer-step counter did not apply"

# i) THE LABELS ARE OFF BY ONE. IN TWO PLACES. This is upstream's bug and it is
#    the reason the loss falls convincingly (12.30 -> 2.32 over 335 updates)
#    while the audio stays unusable and codec_eos is essentially never emitted:
#    it trains a shifted objective, which is learnable and wrong.
#
#    ROOT CAUSE, both times: `self.loss_function` is HuggingFace's, and for these
#    class names transformers resolves it to ForCausalLMLoss, which SHIFTS
#    INTERNALLY -- `labels = pad(labels, (0,1), -100); shift_labels =
#    labels[..., 1:]`. Verified in transformers 4.57.3 (the version qwen-tts
#    pins) and unchanged in 5.12.1. Both call sites hand it labels that are
#    ALREADY aligned, so the shift is applied twice.
#
#    (i.1) TALKER / codec_0. sft_12hz.py feeds inputs_embeds[:, :-1] with
#          labels=codec_0_labels[:, 1:] -- the manual-shift idiom, right for a
#          raw cross_entropy call and wrong when the loss shifts too. Simulated
#          on L = [0..7]:
#              upstream : logits[j] -> L[j+2]     two positions ahead
#              correct  : logits[j] -> L[j+1]
#          So every frame is predicted from the wrong context, and the codec_eos
#          label lands on the second-to-last frame rather than the last -- the
#          model learns to stop from a context that never occurs at inference.
#          Fix: pass FULL-length inputs and UNSHIFTED labels, and let the
#          internal shift be the only shift. Full length rather than
#          labels[:, :-1] because eos sits at the final position of the longest
#          item in a batch, and truncating drops its supervision entirely.
#
#    (i.2) SUB-TALKER / groups 1-15, inside the model file rather than the
#          script. forward_finetune builds `logits[j] = lm_head[j](hidden[j+1])`,
#          and hidden[j+1] has seen codec_0..codec_j, so logits[j] is ALREADY
#          the prediction for group j+1 == labels[j]. loss_function then shifts
#          it onto labels[j+1]. Not reachable by argument, so compute this loss
#          here and discard the one it returns.
_old_fwd = (
    "                outputs = model.talker(\n"
    "                    inputs_embeds=input_embeddings[:, :-1, :],\n"
    "                    attention_mask=attention_mask[:, :-1],\n"
    "                    labels=codec_0_labels[:, 1:],\n"
    "                    output_hidden_states=True\n"
    "                )")
_new_fwd = (
    "                outputs = model.talker(\n"
    "                    inputs_embeds=input_embeddings,\n"
    "                    attention_mask=attention_mask,\n"
    "                    labels=codec_0_labels,\n"
    "                    output_hidden_states=True\n"
    "                )")
assert _old_fwd in s, "(i.1) forward call not found - upstream may have fixed this"
s = s.replace(_old_fwd, _new_fwd)

# hidden_states is full length now. But hidden[j] predicts frame j+1, NOT frame j:
# past_hidden = hidden_states[:, -1:, :] (modeling_qwen3_tts.py:1740) is carried into
# model_kwargs (:1806) and consumed on the NEXT step (:1672) alongside the codec_0
# sampled from THIS step's logits. So the mask must select j where j+1 is a codec
# frame. Selecting j itself leaks the answer -- input_embeddings[j] already contains
# frame j's groups 1..15, which is exactly what the sub-talker must predict, a
# shortcut that does not exist at inference. Upstream issue #337 / PR #278.
assert "                talker_hidden_states = hidden_states[codec_mask[:, :-1]]\n" in s
s = s.replace(
    "                talker_hidden_states = hidden_states[codec_mask[:, :-1]]\n",
    "                _prev_mask = codec_mask.roll(-1, 1)\n"
    "                _prev_mask[:, -1] = False\n"
    "                talker_hidden_states = hidden_states[_prev_mask]\n")

_old_sub = ("                sub_talker_logits, sub_talker_loss = "
            "model.talker.forward_sub_talker_finetune(talker_codec_ids, talker_hidden_states)")
_new_sub = (
    "                sub_talker_logits, _ = "
    "model.talker.forward_sub_talker_finetune(talker_codec_ids, talker_hidden_states)\n"
    "                # direct pairing: logits[j] IS the prediction for group j+1\n"
    "                sub_talker_loss = torch.nn.functional.cross_entropy(\n"
    "                    sub_talker_logits.reshape(-1, sub_talker_logits.size(-1)),\n"
    "                    talker_codec_ids[:, 1:].reshape(-1))")
assert _old_sub in s, "(i.2) sub-talker call not found"
s = s.replace(_old_sub, _new_sub)

assert "codec_0_labels[:, 1:]" not in s, "a pre-shifted codec_0 label survived"
assert "inputs_embeds=input_embeddings,\n" in s, "(i.1) did not apply"
assert "torch.nn.functional.cross_entropy" in s, "(i.2) did not apply"
assert "hidden_states[_prev_mask]" in s, "(i.3) sub-talker pairing did not apply"
assert "hidden_states[codec_mask]" not in s, "(i.3) leaky pairing survived"

# Write LAST. This used to sit after (e), which meant (f) and (g) -- the whole
# LoRA implementation -- edited a string that was never saved: the run did a full
# fine-tune while printing that it was doing LoRA.
p.write_text(s)
assert "flash_attention_2" not in s and "log_with" not in s
assert "_tk.text_projection(" in s


print("optimizer:",
 "AdamW8bit" if "AdamW8bit" in s else "AdamW fp32")
print("sm_%d%d" % torch.cuda.get_device_capability(),
      "| mixed_precision:", "no" if 'mixed_precision="no"' in s else "bf16",
      "| weights:", "fp32" if "torch.float32" in s else "bf16")

# GATE 3. COMPILE THE RESULT. Every assert above checks that a string was
# REPLACED; none checks that what came out is still valid Python. A patch that
# lands with the wrong indentation passes all of them and then dies at step 6 --
# after a 3.7GB base download and a tokenise, ~10 min in, with a traceback
# pointing at a generated file rather than at the patch that generated it.
import py_compile, tempfile
try:
    py_compile.compile("sft_12hz.py", cfile=tempfile.mktemp(), doraise=True)
except py_compile.PyCompileError as e:
    raise SystemExit(f"PATCHED FILE DOES NOT COMPILE:\n{e}")

# And check the two structural facts the asserts above cannot see: that the
# counters landed INSIDE the sync_gradients block (outside it they would tally
# each grad norm 4x and read stale values), and that the save is still guarded.
_src = pathlib.Path("sft_12hz.py").read_text().splitlines()
_sync = next(n for n, l in enumerate(_src) if "if accelerator.sync_gradients:" in l)
_body = _src[_sync + 1:_sync + 6]
assert any("_opt += 1" in l for l in _body), "_opt is not inside the sync_gradients block"
assert any("_bad += 1" in l for l in _body), "_bad is not inside the sync_gradients block"
print("GATE 3 OK -- patched file compiles, counters are in the sync block")


In [ ]:
# 3b. sft_12hz.py does shutil.copytree(MODEL_PATH, ...) at the end of each epoch.
#     MODEL_PATH must therefore be a real local directory, not a hub id -- otherwise
#     it raises FileNotFoundError *after* the epoch has already been trained.
from huggingface_hub import snapshot_download
BASE = snapshot_download("Qwen/Qwen3-TTS-12Hz-0.6B-Base")
print(BASE)
!ls {BASE}
res("after base download")

# GATE 3b. sft_12hz.py copytrees this directory. A partial snapshot_download
# (interrupted, or a cache with a missing blob) copies fine and then produces a
# checkpoint that cannot load -- discovered at step 7, after the whole run.
import pathlib
_base = pathlib.Path(BASE)
assert (_base / "config.json").exists(), "base snapshot has no config.json"
_w = list(_base.rglob("*.safetensors")) + list(_base.rglob("*.bin"))
assert _w, "base snapshot has no weight files"
# 2.0GiB, not 3.0. The full snapshot is ~2.34GiB: model.safetensors 1.70GiB
# (914.6M params in BF16) plus speech_tokenizer/model.safetensors 0.64GiB. The
# 3.7GB figure elsewhere in this notebook is the fp32 checkpoint we WRITE
# (914.6M x 4 bytes = 3.41GiB), not the bf16 model we read -- and a threshold
# taken from the wrong one of those rejects a perfectly good download 10 minutes
# in. Both files must be present, so check them by name too.
assert (_base / "speech_tokenizer" / "model.safetensors").exists(), \
    "speech_tokenizer weights missing from the snapshot"
_gb = sum(f.stat().st_size for f in _base.rglob("*") if f.is_file()) / 2**30
assert _gb > 2.0, f"base snapshot is only {_gb:.2f}GiB -- expected ~2.34GiB, download incomplete"
print(f"GATE 3b OK -- base is {_gb:.2f}GiB, {len(_w)} weight file(s)")


In [ ]:
# 4. Pull the private dataset from HF instead of the browser upload widget.
from huggingface_hub import snapshot_download
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("HF token (hf_...): ")
snapshot_download(DATASET_REPO, repo_type="dataset",
                  local_dir=".", token=os.environ["HF_TOKEN"])
!wc -l train_raw.jsonl && ls wavs | head -3
res("after dataset pull")

# GATE 4. THE 24kHz CHECK IS THE LOAD-BEARING ONE. The 12Hz tokenizer asserts
# sr == 24000 (upstream_dataset.py:105), but a 48kHz dataset has silently reached
# this pipeline before -- and the wavs play back fine, so nothing looks wrong
# until the voice is unusable. Checked here, not after tokenising.
import json as _json, pathlib, soundfile as _sf
rows = [_json.loads(l) for l in open("train_raw.jsonl")]
assert rows, "train_raw.jsonl is empty"
# The key is "audio" -- build_dataset.py:284 writes it and prepare_data.py:46
# reads line['audio'], so asserting the name here also checks compatibility with
# the tokenizer rather than just guessing at a schema.
assert "audio" in rows[0], f"expected an 'audio' key, got {sorted(rows[0])}"
_missing = [r["audio"] for r in rows if not pathlib.Path(r["audio"]).exists()]
assert not _missing, f"{len(_missing)} rows point at wavs that did not download, e.g. {_missing[0]}"

_key = "audio"
_rates, _dur = set(), 0.0
for r in rows:
    _i = _sf.info(r[_key])
    _rates.add(_i.samplerate); _dur += _i.duration
assert _rates == {24000}, f"WRONG SAMPLE RATE {_rates} -- the tokenizer needs 24000 only"
assert pathlib.Path("ref.wav").exists(), "ref.wav missing -- the speaker embedding comes from it"
print(f"GATE 4 OK -- {len(rows)} clips, {_dur/60:.1f} min, all 24kHz, ref.wav present")


In [ ]:
# 5. Extract audio codes with the 12Hz tokenizer.
!python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl train_raw.jsonl \
  --output_jsonl train_with_codes.jsonl
res("after tokenize")


In [ ]:
# 5b. Verify step 5 actually wrote codes before burning GPU time on step 6.
import json
rows = [json.loads(l) for l in open("train_with_codes.jsonl")]
# One coded row per raw row -- that is the invariant, not any fixed count. The
# 29 hardcoded here was the 1.7-min dataset; the aligned scripts give hundreds.
expected = sum(1 for _ in open("train_raw.jsonl"))
assert len(rows) == expected, f"train_raw.jsonl has {expected} rows, got {len(rows)}"
k = next(x for x in rows[0] if "code" in x.lower())
assert all(r[k] for r in rows), "some rows have empty codes"
print(len(rows), "rows | key:", k, "| first row code len:", len(rows[0][k]))


In [ ]:
!ls -la train_with_codes.jsonl && head -c 300 train_with_codes.jsonl

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# 6. LoRA fine-tune, pure fp32 (see patch (c)). Base frozen: no grads/optimizer
#    state for 905.8M params. fp16 autocast measured 8.8GB peak of 14.56GB
#    usable; fp32 grows only the activation share, so expect ~11-13GB. If it
#    OOMs, see the fallback in patch (c).
#
#    LoRA takes a HIGHER lr than full FT (1e-4 vs 2e-5) -- adapters start at zero
#    and have to travel. Full-FT LRs on the Hub are 1e-6..5e-6 for this base;
#    do not copy those here.
#
#    16 EPOCHS, AND THE REASON IS THE ACCUMULATION FACTOR. 167 clips at batch 2
#    is 84 DATALOADER steps per epoch, but sft_12hz.py:71 wraps the body in
#    `with accelerator.accumulate(model)` and gradient_accumulation_steps is 4
#    (line 44, hardcoded), so only 21 of those 84 are real weight updates.
#
#    4 epochs is therefore ~84 updates, not 336. For LoRA adapters initialised at
#    zero -- they start contributing nothing and have to travel -- that is very
#    few; hundreds is the normal range. Which means "loss flattened at 4.79" from
#    the interrupted run, at ~42 updates in, is as consistent with barely-started
#    as with converging. 16 epochs gives 336 updates and makes the curve worth
#    reading. The log now prints `updates:` alongside `Step` so the x-axis is the
#    real one.
#
#    WALL CLOCK: ~2.3s/dataloader-step measured under fp16 autocast (168 steps in
#    385s), so 16 epochs = 1344 steps = ~52 min. This run is pure fp32 (patch c),
#    which gives up the tensor-core matmuls, so expect somewhere in 60-95 min --
#    not measured, and the only number here that is a guess. Either way it is
#    nothing against a 5h30 runtime.
#
#    DISK IS NOW THE BINDING CONSTRAINT, hence patch (g3). Each checkpoint is
#    ~4.4GB (3.7GB safetensors plus the 682MB speech_tokenizer copytree drags
#    in), so saving all 16 wants ~70GB against ~55GB free -- the run would die on
#    disk around epoch 12. (g3) saves every 4th epoch plus the last: epochs
#    3/7/11/15, ~17.6GB, and four points to hear the voice move through.
#
#    The GPU sampler runs in the background because sft_12hz.py is a SUBPROCESS:
#    it releases all its memory on exit, so calling res() afterwards would just
#    report an idle card. Peak is only observable while it runs.
# Clear output/ so no stale directory survives the run. Two reasons: the last
# run left a 2.4GB checkpoint-epoch-1 that died mid-copytree and is not a
# checkpoint at all, and the atomic-save patch (g2) renames into
# checkpoint-epoch-N, which is only truly atomic when nothing is there to delete
# first.
!rm -rf output train.log gpu_train.log
res("before training")
!nohup nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits -l 5 > gpu_train.log 2>&1 &

# -u so the new per-step logs appear live, not in 4KB blocks; tee keeps a copy
# that survives a lost cell output (the first run's log only existed in the
# browser, so the interrupt nearly took the evidence with it).
!python -u sft_12hz.py \
  --init_model_path {BASE} \
  --output_model_path output \
  --train_jsonl train_with_codes.jsonl \
  --batch_size 2 \
  --lr 1e-4 \
  --num_epochs 16 \
  --speaker_name {SPEAKER} 2>&1 | tee train.log

!pkill -f "query-gpu=memory.used,utilization.gpu" || true
import subprocess
rows = [r.split(",") for r in open("gpu_train.log").read().strip().splitlines() if "," in r]
if rows:
    peak = max(int(r[0]) for r in rows)
    print(f"[res] TRAINING PEAK GPU {peak/1024:.1f}GB over {len(rows)} samples "
          f"({len(rows)*5}s), max util {max(int(r[1]) for r in rows)}%")
res("after training")
!du -sh output/checkpoint-epoch-* 2>/dev/null

# GATE 6. DID THE TRAINING ACTUALLY REACH THE CHECKPOINT? This is the one gate
# that could not exist until now, and the most important in the notebook.
#
# Three separate defects in this pipeline all produced a checkpoint that was
# byte-plausible and numerically IDENTICAL TO THE BASE MODEL: peft's
# .base_layer. key infix, .to('cpu') aliasing that unmerge_adapter then rewound,
# and the copytree window where the base model.safetensors is in place and
# save_file has not run yet. Each one loads clean, generates clean, and sounds
# like generic TTS. The asserts in patch (g) run INSIDE training and check key
# names; only comparing tensors against the base catches the values.
#
# q_proj is the probe because LoRA targeted it, so a merged save MUST have moved
# it. codec_embedding would be a false positive -- row 3000 is written
# unconditionally at sft_12hz.py:156 even by a no-op run.
import glob, pathlib
from safetensors import safe_open

# .isdigit() excludes checkpoint-epoch-N.tmp, the half-built directory patch
# (g2) leaves behind when a save is interrupted. The atomic rename means a .tmp
# is BY DEFINITION not a checkpoint -- but it still matches this glob, and
# int("7.tmp") raises ValueError, so the crash-safety patch broke the very cells
# meant to read its output.
cks = sorted((d for d in glob.glob("output/checkpoint-epoch-*")
              if d.rsplit("-", 1)[1].isdigit()),
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "NO CHECKPOINT AT ALL -- training did not finish an epoch"
print("checkpoints:", [c.rsplit("/", 1)[1] for c in cks])

_bw = sorted(pathlib.Path(BASE).rglob("*.safetensors"))
assert _bw, "cannot find base weights to compare against"

for ck in cks:
    _cw = pathlib.Path(ck) / "model.safetensors"
    assert _cw.exists(), f"{ck} has no model.safetensors -- save_file never ran"
    with safe_open(_cw, framework="pt") as c:
        _probe = [k for k in c.keys() if k.endswith("q_proj.weight")][:3]
        assert _probe, f"{ck} has no q_proj weights -- key renaming in (g) went wrong"
        moved = []
        for bf in _bw:
            with safe_open(bf, framework="pt") as b:
                _bk = set(b.keys())
                for k in _probe:
                    if k in _bk:
                        # .float() both sides on purpose. The checkpoint is fp32
                        # and the base is bf16, and Tensor.equal across dtypes is
                        # version-dependent: where it returns False for a dtype
                        # mismatch rather than promoting, `not equal` is always
                        # True and this gate passes a base-identical checkpoint
                        # unconditionally -- the one check covering all three
                        # silent-failure modes, disabled by a torch upgrade.
                        moved.append(not c.get_tensor(k).float()
                                       .equal(b.get_tensor(k).float()))
    assert moved, f"no q_proj key of {ck} was found in the base weights -- cannot verify"
    assert any(moved), (f"{ck} IS THE BASE MODEL. {len(moved)} probed q_proj tensors are "
                        f"bit-identical to the base. Do not score this checkpoint.")
    print(f"  {ck}: {sum(moved)}/{len(moved)} probed q_proj tensors differ from base -- trained")
print("GATE 6 OK -- checkpoints carry trained weights")

# GATE 6b. The curve, against WEIGHT UPDATES rather than dataloader steps -- the
# 4x difference that made 4 epochs look like a longer run than it was. Also
# surfaces bad_grads, which should be 0 throughout in fp32: a non-zero count
# means gradients went non-finite, which is data or LR, not arithmetic.
import re
_log = pathlib.Path("train.log")
if _log.exists():
    _t = _log.read_text()
    pts = [(int(u), float(l)) for l, u in
           re.findall(r"Loss: ([\d.]+) .*?updates: (\d+)", _t)]
    if pts:
        print(f"\nloss vs updates: {pts[0][0]} updates -> {pts[0][1]:.3f} ... "
              f"{pts[-1][0]} updates -> {pts[-1][1]:.3f}")
        for u, l in pts[::max(1, len(pts) // 12)]:
            print(f"  {u:4d} updates  {l:7.3f}  {'#' * int(l * 4)}")
    _bad = [int(b) for b in re.findall(r"bad_grads: (\d+)", _t)]
    if _bad and _bad[-1]:
        print(f"\n!! bad_grads: {_bad[-1]} non-finite grad norms -- weight updates were "
              f"SKIPPED. Not expected in fp32; check the data and the LR.")
    elif _bad:
        print(f"bad_grads: 0 across {len(_bad)} log lines -- every update applied")
    if "NON-FINITE LOSS" in _t:
        print("!! NON-FINITE LOSS appeared -- grep train.log")



In [ ]:
%%writefile gen.py
# 7. Generation runs in a SUBPROCESS on purpose. A CUDA device-side assert tears
#    down the CUDA context for the whole process -- after one fires, every later
#    CUDA call in that same kernel fails, including a fresh from_pretrained on a
#    different checkpoint. In a subprocess the assert kills only the child and the
#    notebook kernel stays healthy, so no runtime restart is ever needed.
import argparse, os, pathlib, time, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel

# THE OLD DEFAULT WAS CONTAMINATED. It was
#   "I have been working on speech models for the last few weeks, mostly on my
#    own laptop."
# which is VERBATIM the first sentence of ref.wav's own transcript
# (recordings/voice_ref3/ref3_text_12s.txt) and a near-paraphrase of training
# passage #1. Scoring on it measures the model echoing its own conditioning.
# Kept below as CONTAMINATED for continuity with older results; never the default.
TEXT = ("Please leave the parcel by the side gate and ring the bell twice "
        "before you go.")
CONTAMINATED = ("I have been working on speech models for the last few weeks, "
                "mostly on my own laptop.")

a = argparse.ArgumentParser()
a.add_argument("ckpt")
a.add_argument("--tag", default="")
a.add_argument("--greedy", action="store_true")
a.add_argument("--cpu", action="store_true")
a.add_argument("--speaker", default=os.environ.get("SPEAKER", "myvoice"))
a.add_argument("--fp32", action="store_true")
# 256 tokens = 21.3s at 12Hz. The benchmark sentence is ~85 chars, which should
# be 5-6s (~72 tokens), so this is 3x headroom -- and it caps the cost of a
# checkpoint that never emits codec_eos (2150). At the old 1024 every runaway
# cost 81.84s of audio and ~3 min of GPU; four of those is 12 min to learn one
# bit of information.
a.add_argument("--max-new", type=int, default=256)
a.add_argument("--text", default=None)
# Zero-shot from the BASE model, conditioned on reference audio instead of a
# trained speaker id -- the baseline, regenerated in the same session on the same
# sentence. Comparing a fine-tune against a number measured in an older session
# is not a comparison.
#
# TWO MODES, AND THEY DIFFER BY 16 POINTS. README records zero-shot ICL at 55.1%
# (spread 1.0pp) and x-vector-only at 39.1% (spread 12.3pp). The branch is
# `use_icl = ref_audio is not None and ref_text is not None`, so ICL needs a
# transcript: without --ref-text this call enters ICL mode with nothing to
# condition on and raises, which is exactly how the first attempt failed.
# 55.1% is the ICL number, so --ref-text is what reproduces the published bar.
a.add_argument("--zeroshot", metavar="REF_WAV", default=None)
a.add_argument("--ref-text", metavar="TXT_FILE", default=None,
               help="transcript of --zeroshot audio; enables ICL mode")
a.add_argument("--xvec", action="store_true",
               help="x-vector-only zero-shot: ignores ref_text (39.1% baseline)")
a = a.parse_args()

if a.cpu:
    dev, dt = "cpu", torch.float32
else:
    # T4 is Turing: is_bf16_supported() says True but only via emulation, which crawls.
    dev = "cuda:0"
    if a.fp32:
        dt = torch.float32   # fp16 generation failed on T4; fp32 inference is only 3.7GB
    else:
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("device", dev, "| dtype", dt, "| greedy", a.greedy)

t0 = time.time()
tts = Qwen3TTSModel.from_pretrained(a.ckpt, device_map=dev, dtype=dt,
                                    attn_implementation="sdpa")
print("  loaded in", round(time.time() - t0), "s")
print("  speakers:", tts.model.get_supported_speakers())

# do_sample=False ALONE IS NOT GREEDY on the 12Hz model. The sub-talker
# (code_predictor) has its own switch, subtalker_dosample, which defaults to True
# independently of do_sample -- inference/qwen3_tts_model.py:325 hard_defaults,
# picked up whenever the caller leaves it None. Its sampled ids then index
# code_predictor.get_input_embeddings()[i] at modeling_qwen3_tts.py:1683, which
# is exactly where an out-of-range index raises a device-side assert.
#
# This matters for the diagnosis, not just correctness: the first run's "greedy
# also failed" looked like proof that bad logits were NOT the cause, since argmax
# cannot emit an out-of-range index whatever the values. But the sub-talker was
# still sampling at top_k=50, temperature=0.9, in fp16. That run was never greedy
# where it mattered, so it rules nothing out.
kw = {"do_sample": False, "subtalker_dosample": False} if a.greedy else {}
TEXT = a.text or TEXT
t1 = time.time()
if a.zeroshot:
    zs = dict(ref_audio=a.zeroshot)
    if a.xvec:
        zs["x_vector_only_mode"] = True
    else:
        assert a.ref_text, "ICL mode needs --ref-text (or pass --xvec)"
        zs["ref_text"] = pathlib.Path(a.ref_text).read_text().strip()
    print("  zero-shot mode:", "x-vector only" if a.xvec else "ICL")
    wavs, sr = tts.generate_voice_clone(text=TEXT, language="English",
                                        max_new_tokens=a.max_new, **zs, **kw)
else:
    wavs, sr = tts.generate_custom_voice(text=TEXT, language="English",
                                         speaker=a.speaker, max_new_tokens=a.max_new, **kw)
# Name from the directory, not from the whole path. BASE is an HF cache
# snapshot (".../snapshots/5d8399...") with no "checkpoint-epoch-N" in it, and
# the old split("-")[-1] turned that into a filename containing slashes.
_name = a.ckpt.rstrip("/").split("/")[-1]
_ck = _name.rsplit("-", 1)[1] if _name.startswith("checkpoint-epoch-") else "base"
out = "ft_" + _ck + a.tag + ".wav"
sf.write(out, wavs[0], sr)
dur = len(wavs[0]) / sr
# DID IT STOP, OR WAS IT CUT OFF? A talker that never emits codec_eos runs to
# max_new_tokens, and the only visible symptom is a suspiciously round duration.
# Within one frame of the cap means it did not terminate on its own -- the audio
# after the sentence is whatever the model does when it has nothing left to say.
# 0.9 of the cap, not the cap exactly. The observed runaway was 81.84s against a
# 1024-token (85.33s) cap -- 982 frames, not 1024 -- so the decoder does not emit
# exactly 12 frames per second and an equality test calls a runaway "stopped".
# A real generation of this sentence is ~5-6s, a quarter of the 256-token cap,
# so the two cases are nowhere near each other and 0.9 separates them cleanly.
cap = a.max_new / 12.0
stopped = dur < 0.9 * cap
print("  wrote", out, round(dur, 2), "s audio in", round(time.time() - t1), "s")
print(f"  {'STOPPED on codec_eos' if stopped else 'RAN TO max_new_tokens -- never emitted eos'}"
      f" ({dur:.2f}s of {cap:.2f}s cap)")
peak = torch.cuda.max_memory_allocated() / 2**30 if dev != "cpu" else 0
import resource
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20   # macOS bytes / linux KB
print(f"  peak GPU {peak:.2f}GB | peak RSS {rss:.2f}GB")


In [ ]:
# 7a. A 2x2, not a fix. The two axes were never separated: every passing run so
#     far was CPU + fp32 + sampled and every failing one GPU + fp16, so device,
#     precision and sampling all moved together.
#
#       --fp32    off/on   -> tests precision
#       --greedy  off/on   -> tests sampling  (BOTH flags, see gen.py)
#
#     THE DECISIVE CELL IS fp16 + greedy. The suspected chain is: fp16 overflows
#     -> non-finite logits -> multinomial on NaN probs (CUDA does not validate)
#     -> a garbage token id -> the embedding lookup at
#     modeling_qwen3_tts.py:1684 asserts. Remove sampling and that chain cannot
#     run, whatever the precision. So fp16+greedy passing confirms the chain end
#     to end; fp16+greedy failing means fp16 breaks something past sampling and
#     precision is the whole story. --fp32 alone would fix the symptom and
#     explain nothing.
#
#     WHY ALL FOUR OF THE FIRST RUN'S ATTEMPTS DIED, INCLUDING BOTH "GREEDY"
#     ONES: gen.py passed only do_sample=False, and subtalker_dosample defaults
#     to True independently (qwen3_tts_model.py:325/346), so the code predictor
#     sampled every time. Those runs were never greedy where it mattered and
#     ruled nothing out. gen.py now sets both.
#
#     CUDA_LAUNCH_BLOCKING=1 because a device-side assert is asynchronous:
#     without it the error surfaces at some unrelated later op with no kernel
#     name, which is exactly the generic message the first run reported. It
#     serialises every launch, so it is slow -- fine for a few short
#     generations, never leave it on for training.
#
#     Grep stderr for "Assertion" rather than tailing it: an assert prints one
#     line per offending block/thread, thousands of them, so the previous
#     stderr[-800:] would have truncated away the one line that names the fault.
import subprocess, glob, os, torch

cks = sorted((d for d in glob.glob("output/checkpoint-epoch-*")
              if d.rsplit("-", 1)[1].isdigit()),   # skip an interrupted *.tmp
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "no checkpoint - did step 6 finish an epoch?"
# Only the fp16 diagnostics get CUDA_LAUNCH_BLOCKING. It serialises every kernel
# launch, which is what makes an async assert report the kernel that actually
# failed -- and pure overhead for the eight fp32 runs, which are the deliverable
# rather than the investigation.
_dbg = {**os.environ, "CUDA_LAUNCH_BLOCKING": "1"}
turing = torch.cuda.get_device_capability()[0] < 8

# The fp16 arm exists only on pre-Ampere; on sm80+ the default is bf16, which
# does not have this failure mode.
runs = []
if turing:
    runs = [("fp16_sampled", [], []), ("fp16_greedy", [], ["--greedy"])]

def go(ck, tag, flags, env=None):
    print("==", ck, tag)
    r = subprocess.run(["python", "gen.py", ck, "--tag", "_" + tag] + flags,
                       capture_output=True, text=True, env=env or os.environ)
    print(r.stdout.strip())
    if r.returncode != 0:
        hits = list(dict.fromkeys(l.strip() for l in r.stderr.splitlines()
                                  if "Assertion" in l or "Error" in l))
        print("  RC", r.returncode)
        print("  " + "\n  ".join(hits[:12]) if hits else r.stderr.strip()[-800:])

for tag, prec, samp in runs:
    go(cks[-1], tag, prec + samp, env=_dbg)

# EVERY checkpoint gets fp32 greedy AND fp32 sampled. The deliverable of this run
# is score-vs-epoch, not a final number: 336 updates on 20 min from one speaker is
# the range where LoRA starts to overfit, and (g3) saved epochs 3/7/11/15 as
# exactly the instrument for finding out. The peak may not be at the end -- if it
# lands at 11 and falls at 15 that is a real result, and scoring only the last one
# cannot tell undertrained from overtrained.
#
# GREEDY IS THE ONE TO SCORE. Sampled generation at temperature 0.9 varies run to
# run, so a sampled score-vs-epoch curve measures sampling luck alongside
# training. Greedy is deterministic: differences between epochs are the weights.
# Sampled is kept because it is what the voice actually sounds like in use.
prec = ["--fp32"] if turing else []
for ck in cks:
    go(ck, "fp32_greedy", prec + ["--greedy"])
    go(ck, "fp32_sampled", prec)

# GATE 7. A wav on disk is not a successful generation. gen.py exits 0 for
# digital silence, for a 0.2s stub, and for a full-length burst of noise -- and
# 7c plays them all through the browser widget where silence is easy to miss.
# Checked here so a dead checkpoint cannot reach timbre_score.py, which would
# happily return a percentage for it.
import glob, numpy as np, soundfile as sf
_out = sorted(glob.glob("ft_*.wav"))
assert _out, "no wav was produced by any run above"
print()
for f in _out:
    y, sr = sf.read(f)
    dur, peak = len(y) / sr, float(np.abs(y).max())
    rms = float(np.sqrt((y.astype("float64") ** 2).mean()))
    # ~85 chars of text: under 2s means it stopped early, over 20s means it ran away
    flags = []
    if peak < 1e-4:            flags.append("SILENT")
    elif rms < 1e-3:           flags.append("near-silent")
    if dur < 2.0:              flags.append("TRUNCATED")
    elif dur > 20.0:           flags.append("RAN AWAY (hit max_new_tokens?)")
    if peak > 0.999:           flags.append("clipping")
    print(f"  {f:34s} {dur:5.2f}s  peak {peak:.3f}  rms {rms:.4f}  "
          f"{'  '.join(flags) or 'ok'}")
# Duration AND level. Filtering on duration alone would pass six seconds of
# digital silence -- which is what a dead checkpoint most often produces -- and
# timbre_score.py returns a percentage for silence rather than refusing it.
# Recomputed here rather than collected in the loop above so the criterion sits
# next to the assert that depends on it.
_good = [f for f in _out
         if sf.info(f).duration >= 2.0 and float(np.abs(sf.read(f)[0]).max()) >= 1e-4]
assert _good, "every generation is silence or a stub -- nothing worth scoring"
print(f"\nGATE 7: {len(_good)}/{len(_out)} usable. Score the _fp32_greedy ones "
      f"(deterministic) for the epoch curve.")


In [ ]:
# 7b. CPU diagnosis. Only worth running if 7a's fp32 runs ALSO failed -- if fp32
#     generated cleanly, precision was the whole story and there is nothing here
#     to find. On CPU, PyTorch raises a real IndexError naming the offending
#     index instead of an async device assert.
#
#     THE OLD HYPOTHESIS HERE WAS WRONG AND IS RECORDED BECAUSE IT LOOKED RIGHT.
#     It said: the talker samples a codec_0 >= 2048 which then indexes
#     code_predictor's embeddings, whose vocab_size is 2048 while the talker's is
#     3072. The vocab numbers are real (configuration_qwen3_tts.py:189 and :373)
#     but codec_0 never touches the 2048 tables. The split is explicit at
#     modeling_qwen3_tts.py:1670 vs :1684 -- codec_0 goes to the talker's OWN
#     3072-row table, and only the code predictor's groups 1..N index the
#     2048-row ones. Same split at :1623/:1625 and :1986/:1988. The predictor's
#     lm_head is Linear(hidden, 2048) (:1167), so its own tokens are
#     structurally in range. Speaker id 3000 lands in the talker's 3072 table,
#     which is exactly where the training patch writes it (sft_12hz.py:156).
#
#     So the ONLY route to an out-of-bounds index at :1684 is multinomial
#     returning garbage from non-finite probs -- i.e. the fp16 chain 7a tests,
#     and nothing else. Which is why 7a, not this cell, is the experiment.
#
#     Pick the checkpoint rather than hardcoding one. `checkpoint-epoch-2` was
#     hardcoded here, and a run interrupted after 2 epochs has only epoch-0 and
#     epoch-1, so this cell 404'd against the Hub -- from_pretrained treats a
#     missing local dir as a repo id -- and the one diagnostic that would have
#     explained the CUDA assert never ran.
#
#     No size filter any more: patch (g2) renames into place, so a checkpoint
#     directory either does not exist or is complete. The filter it replaces had
#     a hole in both directions -- it passed a full-size directory still holding
#     base weights (copytree finished, save_file had not), and with exactly one
#     checkpoint `max` was that checkpoint itself, so a truncated dir cleared its
#     own threshold.
import glob
cks = sorted((d for d in glob.glob("output/checkpoint-epoch-*")
              if d.rsplit("-", 1)[1].isdigit()),   # skip an interrupted *.tmp
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "no checkpoint to diagnose"
CKPT = cks[-1]
print("diagnosing", CKPT)
!python gen.py {CKPT} --tag _cpu --cpu


In [ ]:
# 7c. Listen in the browser before scoring anything.
import glob
from IPython.display import Audio, display
for f in sorted(glob.glob("ft_*.wav")):
    print(f); display(Audio(f))


In [ ]:
# 7d. HELD-OUT TEXT + SAME-SESSION BASELINE.
#
#     Two evaluation defects this fixes.
#
#     (1) gen.py's default sentence is a near-paraphrase of training passage #1
#         ("I have been working on speech models for a few weeks now, mostly in
#         the evenings..."). Not an exact match, but close enough that it scores
#         recall rather than generalisation. HELD_OUT below is about cooking and
#         weather -- no overlap with either script's vocabulary or subject.
#
#     (2) The 55.1% zero-shot bar was measured in an earlier session, on
#         different text, with a different runtime. Comparing a fine-tune to a
#         remembered number is not a comparison. --zeroshot regenerates it HERE,
#         from the same base model, on the same two sentences, in the same
#         session. That is the only baseline worth quoting against.
import subprocess, glob, os, torch

HELD_OUT = ("The kitchen smells of burnt garlic again, and the rain has not "
            "stopped since Thursday morning.")
prec = ["--fp32"] if torch.cuda.get_device_capability()[0] < 8 else []

def run(ckpt, tag, extra):
    r = subprocess.run(["python", "gen.py", ckpt, "--tag", tag, "--greedy"] + prec + extra,
                       capture_output=True, text=True)
    print("==", tag, "|", ckpt.split("/")[-1])
    print(r.stdout.strip())
    # `stdout or stderr` HID A REAL FAILURE. gen.py prints device/dtype/speakers
    # before generating, so stdout is never empty -- which meant the zero-shot
    # runs died after "speakers: dict_keys([])" and the traceback was discarded.
    # Non-zero exit always shows stderr.
    if r.returncode != 0:
        print("  RC", r.returncode, "STDERR:")
        print("   ", "\n    ".join(r.stderr.strip().splitlines()[-15:]))

cks = sorted((d for d in glob.glob("output/checkpoint-epoch-*")
              if d.rsplit("-", 1)[1].isdigit()),
             key=lambda d: int(d.rsplit("-", 1)[1]))
assert cks, "no checkpoint"

# every checkpoint on the held-out sentence
for ck in cks:
    run(ck, "_heldout", ["--text", HELD_OUT])

# THE BASELINE, from the untouched base model. ICL uses a MATCHED PAIR --
# ref_icl.wav is the 12s take with ref_icl.txt its full transcript, so audio and
# text agree. (ref.wav is the first 10s of the same take, so its transcript
# would be a guess at where to cut; x-vector mode ignores text and can use it.)
# Both modes, because README has ICL at 55.1% and x-vector at 39.1% and only the
# first is the published bar.
for tag, extra in [("_zs_icl", ["--zeroshot", "ref_icl.wav",
                                "--ref-text", "ref_icl.txt"]),
                   ("_zs_xvec", ["--zeroshot", "ref.wav", "--xvec"])]:
    run(BASE, tag, extra + ["--text", HELD_OUT])

print("\nwrote:", sorted(f for f in glob.glob("ft_*.wav")
                          if "heldout" in f or "_zs_" in f))


In [ ]:
# 7e-pre. REPAIR, only if a `pip install -U transformers` has already run in
#         this session. Restores the pinned version and proves the import chain
#         works in a SUBPROCESS -- the notebook kernel still holds the broken
#         module objects in sys.modules, so an in-kernel import can keep failing
#         even after the downgrade. gen.py and sft_12hz.py are subprocesses, so
#         the subprocess check is the one that matters.
import subprocess, sys
print(subprocess.run([sys.executable, "-c",
    "import transformers; print('before:', transformers.__version__)"],
    capture_output=True, text=True).stdout.strip() or "before: import fails")

!pip -q install "transformers==4.57.3"

chk = subprocess.run([sys.executable, "-c",
    "import transformers, huggingface_hub as h; from transformers import pipeline; "
    "import qwen_tts; print('OK transformers', transformers.__version__, "
    "'| hub', h.__version__)"], capture_output=True, text=True)
print(chk.stdout.strip())
if chk.returncode != 0:
    print("STILL BROKEN:"); print(chk.stderr.strip()[-800:])
    raise SystemExit("fix the environment before training or generating")
print("7e-pre OK -- subprocesses can import transformers and qwen_tts")


In [ ]:
# 7e. INTELLIGIBILITY. timbre_score.py measures SPEAKER SIMILARITY ONLY -- so
#     confident babble in your own voice scores as your own voice, and an
#     overfit checkpoint can top the timbre curve while being unusable. This is
#     the missing half of the metric: transcribe what was generated and compare
#     it to what was asked for.
#
#     Whisper-small on a T4 is a few seconds per clip. WER by the standard
#     Levenshtein DP over words -- ~15 lines, no extra dependency.
#
#     NO PIP INSTALL. An earlier version of this cell ran
#     `pip install -U transformers[torch]`, which upgraded transformers past the
#     `transformers==4.57.3` that qwen-tts pins, against the huggingface_hub
#     0.36.2 already installed -- giving
#     `ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub'`
#     and breaking every later cell, training included. The install was also
#     pointless: whisper has been in transformers since 4.23, so 4.57.3 already
#     has everything this cell needs. Repair with the cell in 7e-pre if it has
#     already happened.
import glob, re, json, torch
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model="openai/whisper-small",
               device=0 if torch.cuda.is_available() else -1,
               chunk_length_s=30)

def norm(t):
    return re.sub(r"[^a-z0-9 ]", "", t.lower()).split()

def wer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1): d[i][0] = i
    for j in range(len(h) + 1): d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (r[i-1] != h[j-1]))
    return d[-1][-1] / max(1, len(r))

# Must match gen.py and 7d exactly, or WER scores the wrong target.
DEFAULT = ("Please leave the parcel by the side gate and ring the bell twice "
           "before you go.")
HELD_OUT = ("The kitchen smells of burnt garlic again, and the rain has not "
            "stopped since Thursday morning.")

rows = []
for f in sorted(glob.glob("ft_*.wav")):
    target = HELD_OUT if ("heldout" in f or "_zs_" in f) else DEFAULT
    hyp = asr(f)["text"].strip()
    w = wer(target, hyp)
    rows.append({"file": f, "wer": round(w, 3), "heard": hyp})
    print(f"{f:34s} WER {w*100:5.1f}%")
    print(f"    heard: {hyp[:110]}")

json.dump(rows, open("intelligibility.json", "w"), indent=1)
print("\nWER guide: <0.15 usable | 0.15-0.40 degraded | >0.40 not speech you asked for.")
print("A checkpoint that wins on timbre and loses badly here is overfit, not good.")


In [ ]:
# 7f. THE CURVE. One table: does timbre improve with epochs, does intelligibility
#     survive it, and does either beat the same-session zero-shot baseline?
#
#     Read it as a curve, not a leaderboard. A peak at epoch 11 that falls at 15
#     means overfitting -- the answer is fewer epochs or a smaller LoRA rank, not
#     more data. Still climbing at 15 means undertrained -- record script 3.
import json, glob, re
wer = {r["file"]: r["wer"] for r in json.load(open("intelligibility.json"))}

def epoch_of(f):
    m = re.match(r"ft_(\d+|base)_", f)
    return -1 if m.group(1) == "base" else int(m.group(1))

print(f"{'file':34s} {'epoch':>6s} {'WER':>7s}  {'stopped':>8s}")
for f in sorted(glob.glob("ft_*.wav"), key=lambda f: (epoch_of(f), f)):
    import soundfile as sf
    dur = sf.info(f).duration
    print(f"{f:34s} {epoch_of(f):>6} {wer.get(f, float('nan'))*100:>6.1f}% "
          f"{'yes' if dur < 0.9 * 256/12 else 'RAN':>8s}")

print("""
Timbre is scored on the Mac, not here -- the metric's floor voices live in
tts_models/ and its ceiling is calibrated to your reference take:

    cd tts_models/voice_clone/analysis
    ../../.venv/bin/python timbre_score.py ../dataset/finetune_out/finetuned/ft_*.wav

Bar: 55.1% zero-shot (but prefer the ft_base_zs_* files generated above --
same session, same sentence). The 1.7-minute full fine-tune scored 9.5%.""")


In [ ]:
# 8. Push the samples back to HF - files.download() is the same browser-session
#    widget that failed in step 4, so it will not work from a terminal client.
#    Step 4 asked for a READ token; uploading needs WRITE, so ask again here
#    rather than keeping a write token live for the whole session.
import glob, getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass.getpass("HF WRITE token (hf_...): "))
for f in sorted(glob.glob("ft_*.wav")):
    api.upload_file(path_or_fileobj=f, path_in_repo="finetuned/" + f,
                    repo_id=DATASET_REPO, repo_type="dataset")
    print("pushed", f)
# Then on the Mac:
#   hf download $DATASET_REPO --repo-type dataset \
#       --include "finetuned/*" --local-dir tts_models/voice_clone/dataset/finetune_out


## Compare against the full fine-tune

Same metric, same reference (ceiling 0.9956, floor 0.8432):

| | scale |
|---|---|
| zero-shot ICL | **55.1%** |
| zero-shot x-vector | 39.1% |
| Kokoro (floor voice) | 12.7% |
| full FT, 1.7 min, epoch 2 | 9.5% |
| LoRA, 20.0 min, correct objective | ? |

Prefer the `ft_base_zs_icl` / `ft_base_zs_xvec` files that **7d** generates over the
numbers above — same session, same sentence, same runtime.

**Score timbre AND intelligibility.** Timbre alone cannot tell a voice from
confident babble in that voice, which is the failure mode an overfit TTS model
actually has:

```
cd tts_models/voice_clone/analysis
../../.venv/bin/python timbre_score.py ../dataset/finetune_out/finetuned/ft_*.wav
```

Read 7f as a curve, not a leaderboard. Peak at epoch 11 falling at 15 → overfitting,
so fewer epochs or a smaller LoRA rank, **not** more data. Still climbing at 15 →
undertrained, so record script 3.

If LoRA also lands near 10% *on the corrected objective*, the dataset is the binding
constraint rather than the method. The 9.5% run also trained on ASR text at ~14%
WER, so it was never a clean test of full FT either.
